# CNN - MNIST - Tcl Trl

# Import Libraries

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
import os
import sys
sys.path.insert(0,"..")
from utils import *



device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f' The Device is set to : {device}')

# Import dataset

- augmented
- normalized
- padded
- shuffled
- Mnist

In [10]:
trainloader, testloader, trainset, testset = load_mnist(BATCH_SIZE=32,PATH="./data")

Files already downloaded and verified
Files already downloaded and verified


# CNN Model + Tcl Trl

In [11]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.ad_pool = nn.AdaptiveAvgPool2d(output_size=(6,6))
        
        self.fc1 = nn.Linear(in_features=128 * 6 * 6,out_features= 512, bias= True) 
        self.fc2 = nn.Linear(in_features= 512, out_features=256)
        self.fc3 = nn.Linear(256, 10) 
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.ad_pool(x)

        x = x.view(-1, 128 * 6 * 6)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Model : CNN + Tcl Trl

In [12]:
model = CNN().to(device)



num_params = count_param(model)

print("number of parameters:" , num_params)

print(model)

create model


OutOfMemoryError: CUDA out of memory. Tried to allocate 784.00 MiB (GPU 0; 6.00 GiB total capacity; 4.29 GiB already allocated; 0 bytes free; 4.69 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

# Model Train and Evaluation

In [13]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr= 0.1,
                        momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=200)


In [ ]:
def topk(output, target, k):
    correct = 0.0
    batch_size = output.shape[0]
    for sample in range(batch_size):
        topk_sorted = output[sample].sort()[1][:k]
        if target[sample] in topk_sorted:
           correct+=1
        #    print(f'sample {sample} was correct because : {topk_sorted} and {target[sample]}')
    return (correct/batch_size)*100.0
        

In [14]:

def train(epoch):
    file_path = '../results/mnist/tcl_trl/CNN_train.txt'
    
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    
    print('\nEpoch: %d' % epoch)
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    with open(file_path, 'a') as f:
        f.write(f'\nEpoch: {epoch}\n')
        
        for batch_idx, (inputs, targets) in enumerate(trainloader):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

        train_summary = f'Train Summary after Epoch: {epoch}, Loss: {train_loss / len(trainloader):.3f}, Accuracy: {100. * correct / total:.3f}% ({correct}/{total})\n'
        f.write(train_summary)
        print(train_summary)



        model_save_path = f'../results/mnist/tcl_trl/epoch_{epoch}_Cnn_train.pth'
        os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
        torch.save(model.state_dict(), model_save_path)
        print(f'Model saved to {model_save_path}')

In [15]:
def test(epoch):
    file_path = '../results/mnist/tcl_trl/CNN_test.txt'
    
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        with open(file_path, 'a') as f:
            f.write(f'\nTesting after Epoch: {epoch}\n')
            
            for batch_idx, (inputs, targets) in enumerate(testloader):
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)

                test_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()

                # progress_info = f'Loss: {test_loss / (batch_idx + 1):.3f} | Acc: {100. * correct / total:.3f}% ({correct}/{total})'
                # f.write(progress_info + '\n')

                # print(progress_info)
                
            test_summary = f'Test Summary after Epoch {epoch}, Loss: {test_loss / len(testloader):.3f}, Accuracy: {100. * correct / total:.3f}% ({correct}/{total})\n'
            f.write(test_summary)
            
            
            print(test_summary)

In [16]:
Epoch = 300
for epoch in range(1, Epoch + 1):
    train(epoch)
    test(epoch)
    scheduler.step()


Epoch: 0


OutOfMemoryError: CUDA out of memory. Tried to allocate 392.00 MiB (GPU 0; 6.00 GiB total capacity; 4.69 GiB already allocated; 0 bytes free; 5.07 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF